## ANALYTICAL OUTPUTS  (GOLD LAYER)

In [0]:
account_key=dbutils.secrets.get(scope = "My_scope", key = "DB-secret")
# plx = dbutils.secrets.get(scope = "my_scope", key = "sa")
# print(plx)
# dbutils.secrets.listScopes()

In [0]:
storage_account = "storagestartup"
container = "startup-data"


spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    account_key
)

In [0]:
silver_df = spark.read.format("delta").load(
    "abfss://startup-data@storagestartup.dfs.core.windows.net/silver/delta/startup_funding"
)


In [0]:
spark.sql("create database if not exists startup_gold")

DataFrame[]

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("startup_gold.silver_startup")

In [0]:
%sql
SELECT * FROM startup_gold.silver_startup LIMIT 10

Startup,Industry,SubVertical,City,Investors,InvestmentType,InvestmentAmount_USD,Date
Housejoy,EdTech,K12,Mumbai,Lightspeed India,Seed,199000.0,2023-04-19
Groww,Media,Streaming,Bengaluru,IFC,Seed,1668000.0,2025-01-28
Groww,Mobility,Ride Sharing,Hyderabad,"Nexus Venture Partners, Peak XV",Series B,3.8052E7,2021-03-14
FarmBox,Consumer Electronics,Wearables,Gurugram,"Kalaari Capital, Y Combinator",Seed,455000.0,2023-09-11
Udaan,RealEstate,Rental Tech,Mumbai,Bessemer Venture Partners,Seed,89000.0,2024-01-31
AeroStack,EdTech,Coding Bootcamp,Delhi,"Matrix Partners India, Peak XV",Pre-Series A,143000.0,2021-07-24
Ola,Retail,Fashion,Mumbai,Tiger Global Management,Seed,62000.0,2023-01-12
FinSpace,Consumer Electronics,Wearables,Gurugram,Blume Ventures,Growth,4.91721E8,2025-01-22
Rapido,FoodTech,Food Delivery,Kolkata,"A91 Partners, Kedaara Capital, Tiger Global Management",Seed,459000.0,2023-11-27
HyperLoop,EdTech,Test Prep,Ahmedabad,"Mirae Asset, SoftBank Vision Fund, Tiger Global",Series B,3.561E7,2020-05-04


##1.top_funded_sectors

**Business Question:** Which sectors attracted the highest cumulative investment?

**Technique:** GROUP BY + SUM + ORDER BY

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE startup_gold.top_funded_sectors
USING DELTA AS
SELECT Industry, SUM(InvestmentAmount_USD) AS TotalFunding
FROM startup_gold.silver_startup
GROUP BY Industry
ORDER BY TotalFunding DESC
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT *FROM startup_gold.top_funded_sectors
""").show()

+--------------------+------------+
|            Industry|TotalFunding|
+--------------------+------------+
|            FoodTech|  3.477282E9|
|Consumer Electronics|  2.703489E9|
|              Retail|  2.673074E9|
|            Mobility|  2.460096E9|
|               Media|  2.292449E9|
|            AgriTech|  2.142559E9|
|          E-commerce|  1.772544E9|
|             FinTech|  1.631375E9|
|                SaaS|  1.629119E9|
|              EdTech|  1.627226E9|
|          HealthTech|  1.623517E9|
|          Enterprise|  1.414032E9|
|           Logistics|  1.372049E9|
|          RealEstate|  1.267432E9|
+--------------------+------------+



## 2.city_funding_rank
**Business Question:** Which cities beyond Tier-1 are 
emerging as startup hubs?

**Technique:** RANK() OVER (ORDER BY 
total_funding)  

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE startup_gold.city_funding_rank
USING DELTA AS

WITH city_summary AS
(
    SELECT
        City,
        SUM(InvestmentAmount_USD) AS TotalFunding
    FROM startup_gold.silver_startup
    GROUP BY City
)

SELECT
    City,
    TotalFunding,
    RANK() OVER (
        ORDER BY TotalFunding DESC
    ) AS FundingRank

FROM city_summary
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
spark.sql("""
SELECT *FROM startup_gold.city_funding_rank ORDER BY FundingRank
    """)
)

City,TotalFunding,FundingRank
Pune,4.435496E9,1
Kolkata,3.590477E9,2
Delhi,3.426493E9,3
Gurugram,3.122328E9,4
Chennai,2.675023E9,5
Bengaluru,2.561458E9,6
Ahmedabad,2.462396E9,7
Mumbai,2.424807E9,8
Hyderabad,1.707975E9,9
Noida,1.67979E9,10


## 3. sector_yoy_snapshot 
**Business Question:** How has EdTech / FinTech funding 
shifted post-COVID?
**Technique:** SCD Type 2 MERGE + CTE

In [0]:
spark.sql("DROP TABLE IF EXISTS startup_gold.sector_yoy_snapshot")

spark.sql("""
CREATE TABLE startup_gold.sector_yoy_snapshot (
    Industry STRING,
    FundingYear INT,
    TotalFunding DOUBLE,
    StartDate DATE,
    EndDate DATE,
    IsCurrent BOOLEAN
)
USING DELTA
""")

DataFrame[]

In [0]:
spark.sql("""
INSERT INTO startup_gold.sector_yoy_snapshot
SELECT
    Industry,
    YEAR(Date) AS FundingYear,
    SUM(InvestmentAmount_USD) AS TotalFunding,
    current_date(),
    NULL,
    TRUE
FROM startup_gold.silver_startup
GROUP BY Industry, YEAR(Date)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
spark.sql("SELECT * FROM startup_gold.sector_yoy_snapshot ORDER BY Industry, FundingYear")
)

Industry,FundingYear,TotalFunding,StartDate,EndDate,IsCurrent
AgriTech,2020,6.0977E8,2026-07-10,null,true
AgriTech,2021,2.96223E8,2026-07-10,null,true
AgriTech,2022,1.34158E8,2026-07-10,null,true
AgriTech,2023,1.91129E8,2026-07-10,null,true
AgriTech,2024,5.24078E8,2026-07-10,null,true
AgriTech,2025,3.87201E8,2026-07-10,null,true
Consumer Electronics,2020,1.82886E8,2026-07-10,null,true
Consumer Electronics,2021,7.50328E8,2026-07-10,null,true
Consumer Electronics,2022,1.57918E8,2026-07-10,null,true
Consumer Electronics,2023,7.1029E7,2026-07-10,null,true


In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW sector_updates AS
SELECT
    Industry,
    YEAR(Date) AS FundingYear,
    SUM(InvestmentAmount_USD) AS TotalFunding
FROM startup_gold.silver_startup
GROUP BY Industry, YEAR(Date)
""")

DataFrame[]

In [0]:
spark.sql("""
MERGE INTO startup_gold.sector_yoy_snapshot AS target
USING sector_updates AS source
ON target.Industry = source.Industry
AND target.FundingYear = source.FundingYear
AND target.IsCurrent = TRUE
WHEN MATCHED AND target.TotalFunding <> source.TotalFunding
THEN UPDATE SET
    target.EndDate = current_date(),
    target.IsCurrent = FALSE
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
INSERT INTO startup_gold.sector_yoy_snapshot
SELECT
    source.Industry,
    source.FundingYear,
    source.TotalFunding,
    current_date(),
    NULL,
    TRUE
FROM sector_updates source
LEFT JOIN startup_gold.sector_yoy_snapshot target
    ON source.Industry = target.Industry
    AND source.FundingYear = target.FundingYear
    AND target.IsCurrent = TRUE
WHERE target.Industry IS NULL
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
spark.sql("""
SELECT * FROM startup_gold.sector_yoy_snapshot
ORDER BY Industry, FundingYear, StartDate
""")
)

Industry,FundingYear,TotalFunding,StartDate,EndDate,IsCurrent
AgriTech,2020,6.0977E8,2026-07-10,null,true
AgriTech,2021,2.96223E8,2026-07-10,null,true
AgriTech,2022,1.34158E8,2026-07-10,null,true
AgriTech,2023,1.91129E8,2026-07-10,null,true
AgriTech,2024,5.24078E8,2026-07-10,null,true
AgriTech,2025,3.87201E8,2026-07-10,null,true
Consumer Electronics,2020,1.82886E8,2026-07-10,null,true
Consumer Electronics,2021,7.50328E8,2026-07-10,null,true
Consumer Electronics,2022,1.57918E8,2026-07-10,null,true
Consumer Electronics,2023,7.1029E7,2026-07-10,null,true


In [0]:
%sql
SELECT
    Industry,
    FundingYear,
    TotalFunding
FROM startup_gold.sector_yoy_snapshot
WHERE IsCurrent = TRUE
  AND Industry IN ('EdTech', 'FinTech')
  AND FundingYear >= 2020
ORDER BY Industry, FundingYear

Industry,FundingYear,TotalFunding
EdTech,2020,3.77572E8
EdTech,2021,3.3978E7
EdTech,2022,2.08802E8
EdTech,2023,2.16741E8
EdTech,2024,7.10206E8
EdTech,2025,7.9927E7
FinTech,2020,2.33824E8
FinTech,2021,5.23515E8
FinTech,2022,1.70109E8
FinTech,2023,3.58121E8


In [0]:
spark.sql("""
SELECT Industry, FundingYear, COUNT(*) 
FROM startup_gold.sector_yoy_snapshot
GROUP BY Industry, FundingYear
HAVING COUNT(*) > 1 """)

DataFrame[Industry: string, FundingYear: int, count(1): bigint]

## 4. investor_deal_count
**Business Question:** Who are the most active investors by 
volume? 

**Technique:**  GROUP BY + COUNT 

In [0]:
spark.sql("""

CREATE OR REPLACE TABLE startup_gold.investor_deal_count
USING DELTA AS SELECT Investors, COUNT(*) AS TotalDeals
FROM startup_gold.silver_startup GROUP BY Investors
ORDER BY TotalDeals DESC

""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
spark.sql("""
SELECT *

FROM startup_gold.investor_deal_count

"""))

Investors,TotalDeals
Info Edge,29
Zodius Capital,27
Kalaari Capital,26
Elevation Capital,25
Y Combinator,25
Accel,24
Sequoia Capital India,24
Mirae Asset,23
Blume Ventures,22
Tiger Global Management,22


## 5. avg_deal_by_stage 
**Business Question:** What is the average deal size at Seed vs 
Series A vs Series B?

**Technique:** AVG() + CASE WHEN + GROUP BY

In [0]:
spark.sql("""

CREATE OR REPLACE TABLE startup_gold.avg_deal_by_stage
USING DELTA AS

SELECT

CASE
    WHEN InvestmentType = 'Seed' THEN 'Seed'
    WHEN InvestmentType = 'Series A' THEN 'Series A'
    WHEN InvestmentType = 'Series B' THEN 'Series B'
    ELSE 'Other'
END AS InvestmentStage,

AVG(InvestmentAmount_USD) AS AvgDealAmount

FROM startup_gold.silver_startup

GROUP BY

CASE
    WHEN InvestmentType = 'Seed' THEN 'Seed'
    WHEN InvestmentType = 'Series A' THEN 'Series A'
    WHEN InvestmentType = 'Series B' THEN 'Series B'
    ELSE 'Other'
END

ORDER BY AvgDealAmount DESC

""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
spark.sql("""
SELECT *
FROM startup_gold.avg_deal_by_stage
"""))

InvestmentStage,AvgDealAmount
Other,5.7529434146341465E7
Series B,2.9063745454545453E7
Series A,4960330.3964757705
Seed,499059.49008498585


In [0]:
spark.sql("SHOW TABLES IN startup_gold").show(truncate=False)

+------------+-------------------+-----------+
|database    |tableName          |isTemporary|
+------------+-------------------+-----------+
|startup_gold|avg_deal_by_stage  |false      |
|startup_gold|city_funding_rank  |false      |
|startup_gold|investor_deal_count|false      |
|startup_gold|sector_yoy_snapshot|false      |
|startup_gold|silver_startup     |false      |
|startup_gold|top_funded_sectors |false      |
|            |sector_updates     |true       |
+------------+-------------------+-----------+



#Gold Layer Summary

The Gold layer represents the final analytical layer of the Medallion Architecture, where the cleansed Silver data is transformed into business-ready datasets for reporting and decision-making. In this notebook, SQL-based transformations were implemented to create multiple Delta tables that answer key business questions related to startup funding trends.

The implemented analytics include:

->Sector-wise funding analysis to identify the highest-funded industries.
City-wise investment analysis to rank cities based on total funding received.

->Investor deal count analysis to identify the most active investors by the number of funding deals.

->Funding stage analysis to understand funding distribution across different investment stages.

->Year-over-Year (YoY) sector funding analysis with SCD Type 2 implementation to preserve historical records and track changes over time.

The Gold layer makes extensive use of SQL transformations, aggregation functions (COUNT, SUM, AVG), GROUP BY, ORDER BY, joins where appropriate, and Delta Lake tables to produce optimized, query-ready datasets. Each analytical result is stored as an independent Delta table to support downstream reporting, business intelligence, and dashboarding. This layer completes the transformation of raw startup funding data into meaningful business insights, fulfilling the analytical objectives defined in the project documentation.


The only thing it does not contain is a JOIN, because all the required information comes from a single table

From a SQL and data engineering perspective, adding a JOIN here would be unnecessary. Joins are used when data needs to be combined from two or more related tables. Since Investors already exists in silver_startup, grouping directly on that table is the correct and most efficient approach according to my Knowledge.